In [1]:
import time

start_time = time.time()
print(f"Start time recorded: {start_time}")

Start time recorded: 1763014795.4815695


<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [2]:
do = False # @param{type:"boolean"}
if do:
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

In [3]:
if do:
    !git clone https://github.com/phonchi/CryoParticleSegment.git

In [4]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

In [5]:
if do:
    #%git clone https://github.com/netw0rkf10w/CRF.git
    %cd CryoParticleSegment/Modeling/CRF_main
    !python setup.py clean --all
    !rm -rf build/
    !python setup.py build_ext --inplace --force
    !python setup.py install

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [6]:
if do:
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [7]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/results_50_CRF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
IMAGE_DIR = "/content/image_dir"

In [11]:
%cd /content/

/content


### ✅ Packages Handling

In [12]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [13]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [14]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [16]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = True # @param {type:"boolean"}


In [17]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [18]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [19]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [20]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
train_dataset =  MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset , batch_size=None, shuffle=False, pin_memory=True)

val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True)

full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True)

In [21]:
shape = None
for i1, i2, i3, i4 in val_loader: #test loader and reconstruct
    shape = i4.shape
    break

In [22]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [23]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:27<00:00,  3.05it/s]


In [24]:
processed_micrographs.shape

(84, 3840, 3840)

In [25]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [26]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [27]:
del model
torch.cuda.empty_cache()

In [28]:
model = model_post

In [29]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/results_50_CRF/10017/unet_eb5_dice_CRF'

In [30]:
import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()
        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

Output hidden; open in https://colab.research.google.com to view.

In [31]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [32]:
!cp {RESULT_DIR}/label_images.npy .

In [33]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [34]:
label_images.shape

(84, 3840, 3840)

In [35]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [36]:
!cp {RESULT_DIR}/best_config.txt .

In [37]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

### Actual seg

In [38]:
radius = RADIUS

In [39]:
from skimage import filters, segmentation, morphology
import skimage as ski
from skimage import img_as_float
from center_finding import normalize, min_rect_circle, eliminate_near
import cv2

In [40]:
from skimage import img_as_float
from skimage.filters import threshold_local

cv_list = []
for img in label_images:
    output_image = img_as_float(img)
    block_size = int(round(radius * 1.6)) | 1  # Ensure it's an odd number
    local_thresh = filters.threshold_local(output_image, block_size, method='gaussian', offset=0)
    image_seg = output_image > local_thresh
    thresh1 = ski.util.img_as_ubyte(image_seg)

    contr_min = radius*cv_config[1]
    kernel_size = int(radius / cv_config[0])  # Example ratio
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    thresh1 = cv2.erode(thresh1, kernel, iterations=1)

    contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_array = [c for c in contours]

    # Filter out small/large region and find bounding box with center
    c_ = np.array([cv2.contourArea(contour) for contour in contours])
    aa = (c_>contr_min) & (c_<500000)
    aa = aa.tolist()
    c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
    c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
    c_list = [x for x in c_list if x is not None]
    cv_list.append(c_list)
    print(len(c_full_list), len(c_list))

536 449
549 476
509 395
375 354
388 356
503 428
378 365
481 439
536 444
582 497
538 486
521 485
537 464
478 442
575 484
542 467
506 433
517 457
488 434
577 507
556 461
525 436
510 436
524 404
487 379
488 389
504 405
554 446
505 392
548 492
547 475
487 375
532 440
550 464
544 470
546 453
528 453
537 437
544 485
520 423
416 372
542 447
563 503
374 351
301 295
402 362
348 337
526 493
562 487
490 398
529 450
528 448
566 512
525 497
555 491
484 435
525 447
555 482
479 383
482 396
448 433
491 427
488 457
473 438
464 422
463 437
403 385
459 400
404 344
449 383
459 399
223 223
461 407
488 438
399 354
496 421
416 396
472 411
463 418
354 344
465 402
463 402
414 354
463 398


In [41]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    #if idx == 6:
    #    break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in cv_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [42]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [43]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in cv_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [44]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,3530,3950,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1511,3951,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,431,3950,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,1238,3948,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,248,3947,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
35651,1815,150,00393@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35652,315,148,00394@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35653,2854,155,00395@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35654,1981,174,00396@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [45]:
import starfile
starfile.write(df, f'{RESULT_DIR}/cv_particles.star')

### TrackPy

In [46]:
cv_list

[[(3402, 3822),
  (1383, 3823),
  (303, 3822),
  (1110, 3820),
  (120, 3819),
  (623, 3813),
  (3583, 3814),
  (2410, 3815),
  (3040, 3807),
  (3686, 3782),
  (1186, 3801),
  (1795, 3780),
  (2823, 3783),
  (2693, 3789),
  (1564, 3780),
  (1698, 3763),
  (2952, 3761),
  (975, 3728),
  (1295, 3729),
  (3421, 3689),
  (27, 3688),
  (468, 3685),
  (366, 3672),
  (1782, 3681),
  (1136, 3653),
  (1532, 3653),
  (2769, 3658),
  (1435, 3665),
  (3054, 3648),
  (16, 3607),
  (3694, 3573),
  (992, 3594),
  (754, 3565),
  (3227, 3547),
  (3089, 3554),
  (1127, 3538),
  (37, 3532),
  (879, 3506),
  (675, 3518),
  (2365, 3490),
  (138, 3485),
  (2999, 3493),
  (1577, 3476),
  (2776, 3459),
  (602, 3469),
  (3786, 3447),
  (1773, 3459),
  (765, 3431),
  (1479, 3436),
  (2265, 3425),
  (999, 3414),
  (16, 3408),
  (2931, 3403),
  (2644, 3413),
  (516, 3409),
  (3148, 3373),
  (2808, 3359),
  (673, 3350),
  (209, 3358),
  (884, 3369),
  (1597, 3339),
  (2144, 3330),
  (3372, 3321),
  (331, 3312),
  (

In [47]:
tp_config

(0.25, 0.4)

In [48]:
import trackpy as tp

In [49]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = round(TrackParticleSize*tp_config[1])
scale = tp_config[0]  # Scale factor (0.5 means reducing the size to half)

In [50]:
tp_list = []
for img in label_images:
    output_image = img
    small_image = cv2.resize(output_image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    # Adjust parameters based on the scale
    small_sep = int(sep * scale)
    small_diameter = int(TrackParticleSize * scale)
    if small_diameter % 2 == 0:
        small_diameter += 1
    coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
    #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
    coorTrack['x'] *= (1/scale)
    coorTrack['y'] *= (1/scale)
    coords = coorTrack[['x', 'y']].values
    tp_list.append(coords)
    print(len(coords))

618
595
663
355
396
581
366
494
632
645
560
505
605
496
679
607
563
556
503
602
629
646
613
712
612
614
628
668
638
581
595
650
623
604
609
656
589
650
580
641
440
658
596
358
273
423
318
503
635
591
612
608
588
509
582
504
596
614
640
603
409
524
480
465
458
437
384
508
462
503
508
173
494
499
438
556
394
512
458
317
505
504
453
485


In [51]:
tp_list

[array([[1410.71598755,   63.06070905],
        [ 709.62706015,   93.14249054],
        [1010.71804703,   55.49054192],
        ...,
        [2034.56892418, 3750.14726586],
        [2698.26066336, 3777.33401698],
        [3684.32213121, 3778.60460844]]),
 array([[1978.33010468,   42.72815073],
        [2197.74021323,   54.21038496],
        [  85.98407938,   83.16435262],
        ...,
        [ 472.76550267, 3762.84470403],
        [1674.08774373, 3784.79294336],
        [2778.21281589, 3782.62998198]]),
 array([[ 288.54904646,   59.36613166],
        [ 923.0413086 ,   82.03322863],
        [2100.67181186,   55.22319706],
        ...,
        [1244.70853754, 3786.91668383],
        [2746.80252149, 3780.3339064 ],
        [3689.21310824, 3779.25910626]]),
 array([[1258.35277059,   67.9265044 ],
        [2248.0585921 ,   52.28242562],
        [1065.74275194,  134.77632002],
        [1748.31943948,   97.22520912],
        [2389.80184237,   93.71783009],
        [3447.88511568,  173.358201

In [52]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in tp_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [53]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in tp_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [54]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1538.715988,191.060709,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,837.627060,221.142491,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,1138.718047,183.490542,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,3059.542685,188.432531,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,1469.926443,223.856439,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
45131,1623.702271,3861.517504,00480@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
45132,3369.809722,3911.192226,00481@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
45133,3449.998075,3855.981688,00482@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
45134,1527.914020,3905.324243,00483@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [55]:
import starfile
starfile.write(df, f'{RESULT_DIR}/tp_particles.star')

### Nonmax

In [56]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [57]:
nms_config

(0.5, 0.6)

In [58]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more
pCan=nms_config[0] #the grid size smaller will generate more candidate
overlap=nms_config[1] #allow for overlap
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [59]:
nms_list = []
for idx, img in enumerate(label_images):
    # if idx == 6:
    #     break
    heatArr=pad_image(img)
    heatArr=reNorm(heatArr, nSep)
    heatArr=reLev(heatArr,level)
    gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

    # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
    gsize=psize*pCan
    heat=heatArr.astype(np.float32)
    gaus=gaus.astype(np.float32)
    score=np.zeros(heat.shape).astype(np.float32)
    [sizex,sizey]=heat.shape
    sizex = int(sizex)
    sizey = int(sizey)
    psize = int(psize)
    gsize = int(gsize)

    func = scoreGpu

    tx=16
    ty=16
    bx=(sizex-1)//tx+1
    by=(sizey-1)//ty+1
    print('get score gpu',tx, ty, bx, by, gsize, psize)


    func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
        block=(tx, ty, 1), grid=(int(bx), int(by)))

    # # Calculate necessary dimensions and convert all to int
    numx = int((sizex - 1) // gsize + 1)
    numy = int((sizey - 1) // gsize + 1)
    num = numx * numy
    tnum = num * 5

    # # Create the result array
    res = np.zeros(tnum).astype(np.float32)

    # # Block and grid dimensions
    bx = (numx - 1) // tx + 1
    by = (numy - 1) // ty + 1

    # Get the function from the module
    func = getMax

    # Call the function, ensure all parameters are the correct type
    func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
        block=(16, 16, 1), grid=(bx, by, 1))

    # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
    niter = np.int32(nIter)
    gsize = np.int32(gsize)
    sizex = np.int32(sizex)
    sizey = np.int32(sizey)
    numx = np.int32(numx)
    tnum = np.int32(num)

    print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
    func = getMax3
    # Ensuring the correct parameter order and type for the kernel invocation
    func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
            block=(16, 16, 1), grid=(bx, by, 1))

    canList=reshape(res,num)

    print('Number of Particles before:', len(canList))
    if(len(canList)>pnum):
        canList=canList[:pnum]


    canList=cleanCanList(canList,overlap,psize)
    #canList=reCan(canList,ratio)
    print('Number of Particles:', len(canList))
    nms_list.append([(r[1],r[0]) for r in canList])

get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3475
Number of Particles: 659
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3476
Number of Particles: 613
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3482
Number of Particles: 699
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3211
Number of Particles: 463
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3262
Number of Particles: 449
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3447
Number of Particles: 629
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3

In [60]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
    # else:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in nms_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [61]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in nms_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [62]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,347.8,2891.8,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,2302.4,3760.0,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,2800.0,3095.8,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,2457.0,359.0,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,1138.4,1795.2,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
49501,558.0,3903.8,00564@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
49502,1449.0,3264.0,00565@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
49503,2752.0,2504.0,00566@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
49504,2048.0,3328.0,00567@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [63]:
starfile.write(df, f'{RESULT_DIR}/nms_particles.star')

In [64]:
# @markdown ---
# @markdown time used
end_time = time.time()
print(f"End time recorded: {end_time}")

elapsed_time = end_time - start_time
elapsed_time = elapsed_time


hours = int(elapsed_time // 3600)
remaining_seconds = elapsed_time % 3600

minutes = int(remaining_seconds // 60)
seconds = round(remaining_seconds % 60, 3)

print(f"Time spend : {hours} h, {minutes} m, {seconds} s")


gpu_used = "L4" # @param ["CPU high", "T4", "T4 high", "L4"]
per_unit_cost_dict = {"L4" : 1.71, "T4 high" : 1.41, "T4" : 1.19, "CPU high" :  0.24}
per_unit_cost = per_unit_cost_dict[gpu_used]
print(f"unit price per hr {per_unit_cost}")

cost_units = per_unit_cost * elapsed_time / 3600

per_unit_US = 10.49 / 100

cost_price_US = cost_units * per_unit_US

print(f"unit cost : {round(cost_units, 4)}")
print(f"unit price US: {cost_price_US}")
print(f"unit price NTD: {cost_price_US * 30.76}")

End time recorded: 1763016012.6490266
Time spend : 0 h, 20 m, 17.167 s
unit price per hr 1.71
unit cost : 0.5782
unit price US: 0.060648411468836066
unit price NTD: 1.8655451367813976
